# 如何使用此代码库解决新问题？

该代码库支持三种不同的问题：

* Turán 问题：对于一个包含 `N` 个节点的无向图，在没有长度为 4 的环的情况下，最多可以有多少条边？
* 网格上无等腰三角形：在 `N x N` 的网格中，最多可以放置多少个点，使得其中任意三个点不构成（可能是退化的）等腰三角形？
* 球面上无 5 点问题：在 `N x N x N` 的三维网格中，最多可以放置多少个点，使得其中任意 5 个点不共球面？

当然，您可能对其他问题也感兴趣。本笔记本将指导您如何创建一个新的环境用于训练。

## 如何创建新环境？

以下是在流程中实现您自己的数学问题的逐步指南。

1. 在 `src/envs/` 目录中为您的环境创建一个文件（例如 `my_env.py`），并使用以下（最小化）导入：
```
from src.envs.environment import DataPoint, BaseEnvironment
from src.envs.tokenizers import SparseTokenizer, DenseTokenizer
from src.utils import bool_flag
```
您也可以复制现有文件之一（例如 `isosceles.py`）并在此基础上进行修改。

1. 在此文件中，定义与您想要研究的数学对象相对应的类。这应该是一个 DataPoint 类，包含：
   - `__init__`：初始化数据结构。
   - `calc_score`：计算目标函数值（如果无效则为 -1，例如当某些约束不满足时；如果有效则为大于等于 0，且越大越好）
   - `calc_features`：（可选）在数学对象和字符串之间创建一对一映射，用于去重
   - `local_search`：提供一个算法，可以：(i) 修复您的 DataPoint（如果无效）(ii) （可选）尝试修改它以改进其分数
   - `_batch_generate_and_score`：提供一个方法来生成随机（有效）的数学对象实例


2. 为您的环境创建 Tokenizer。对此有两种选择：
  - 选项 1：如果您的数学对象可以用 k-坐标点序列表示，那么您可以跳过此步骤，并使用 `tokenizers.py` 文件中现有的 tokenizer 之一。
  - 选项 2：如果不能，或者如果您想拥有自己的 tokenizer，您可以基于 `tokenizers.py` 文件中的 `Tokenizer` 创建一个类。它应包含：
    - 一个 `encode` 方法，接收一个 DataPoint 实例并将其（一对一地）表示为 token 列表
    - 一个 `decode` 方法，接收 token 列表并重新创建相应的 DataPoint

2. 定义 Environment 类本身，基于 BaseEnvironment，并指定：
   - `data_class`：对您的 DataPoint 类的引用。这应该在新的 Environment 类的 `__init__` 之前指定。
   - 在新的 Environment 类的 `__init__` 中使用一个 tokenizer（见上一步）。如果您选择了选项 1，可以使用 `tokenizers.py` 中已经实现的类之一，您只需在新的 Environment 类的 `__init__` 之前额外指定：
      - `k`：每个元素索引的维度（例如图有 `k=2`，因为它可以用二维点序列表示，每个点代表一条边）
      - `are_coordinates_symmetric`：序列元素的索引是否可以置换（例如对于无向图为 `True`）
   - 一个 `register_args` 方法，用于指定命令行参数，对应于特定于您的数学环境的附加参数（例如不同的生成方法、使用的最大整数等），以及它们的默认值。

3. 通过将您的环境添加到字典 `ENVS` 中来在 `envs/__init__.py` 中注册它。（例如 `MyEnv:"my_env"`）。

现在您可以通过将 `env_name` 设置为您在 `ENVS` 字典中提供的键（例如 `my_env`）来在此问题上使用此流程。



### 示例 1：{0, ..., `N`-1} 中无 3 项等差数列


给定集合 {0, 1, 2, ..., N-1}，找到最大的子集，使得其中没有三个元素构成等差数列。

#### 步骤 1：构建 DataPoint 结构

DataPoint 负责将示例信息存储在 `self.data` 中。此外，DataPoint 具有运行 `local_search()` 的方法，从而改进当前的数据点。

In [ ]:
import numpy as np
from itertools import combinations
from src.envs.environment import DataPoint


class ArithmeticProgressionFreeDataPoint(DataPoint): # <- New class representing the mathematical object, based on DataPoint
    """
    Represents an Arithmetic Progression-free subset of {0, 1, ..., N-1}.
    
    Attributes:
        N: Universe size {0, 1, ..., N-1}
        data: Binary array, data[i] = 1 means element i is selected
        progressions: List of arithmetic progressions (violations, needed for local search)
        score: Number of elements if valid, -1 otherwise
        features: String representation
    """
    
    def __init__(self, N, init=False):
        self.N = N
        self.data = np.zeros(N, dtype=np.uint8)
        self.progressions = []        
        if init: # <- Required by the pipeline; If true, we want to create an element at random. This will be used during the process to differenciate between generation and decoding of the output of the neural network.
            self._add_elements_greedily() # <- Method to create a valid construction at random.
            self.calc_features() 
            self.calc_score()
    
    def get_elements(self): # <- Method specific to the problem.
        """
        Return set of selected elements. 
        """
        return set([i for i in range(self.N) if self.data[i] == 1])
    
    def calc_score(self): # <- Method required by the pipeline
        """
        Calculate the score.
        Score = number of elements if no arithmetic progression exists, -1 otherwise.
        """
        if len(self.progressions) > 0:
            self.score = -1
        else:
            self.score = int(self.data.sum())
    
    def calc_features(self): # <- Optional method used by the pipeline
        """
        Create a binary string representation for deduplication.
        """
        self.features = ",".join(str(self.data[i]) for i in range(self.N))
    
    def _would_create_ap(self, new_element, existing_set): # <- Method specific to the problem.
        """
        Check if adding new_element would create an arithmetic progression.
        """
        # Case 1: new_element is the middle (b)
        # We need a, c such that a + c = 2 * new_element
        target = 2 * new_element
        for a in existing_set:
            c = target - a
            if c != a and c in existing_set:
                # Found a, c such that a + c = 2 * new_element
                return True
        
        # Case 2: new_element is an endpoint
        for b in existing_set:
            c = 2 * b - new_element
            if c != new_element and c != b and c in existing_set:
                return True
            
        return False
    
    def _add_elements_greedily(self): # <- Method specific to the problem.
        """
        Greedily add elements while avoiding arithmetic progressions.
        """
        order = list(range(self.N))
        np.random.shuffle(order)
        
        current_set = self.get_elements()
        
        for elem in order:
            if not self._would_create_ap(elem, current_set):
                self.data[elem] = 1
                current_set.add(elem)
    
    def _compute_progressions(self): # <- Method specific to the problem.
        """
        Find all arithmetic progressions in the current selection.
        """
        elements = self.get_elements()
        element_set = set(elements)
        self.progressions = []
        
        for a, c in combinations(elements, 2):
            if (a + c) % 2 == 0:
                b = (a + c) // 2
                if b in element_set and a < b < c:
                    self.progressions.append((a, b, c))
    
    def _remove_elements_greedily(self): # <- Method specific to the problem.
        """
        Greedily remove elements to eliminate all arithmetic progressions.
        """
        while self.progressions:
            # Count how many arithmetic progressions each element appears in
            element_count = {}
            for ap in self.progressions:
                for elem in ap:
                    element_count[elem] = element_count.get(elem, 0) + 1
            
            # Remove the element in the most arithmetic progressions
            worst_element = max(element_count, key=element_count.get)
            self.data[worst_element] = 0
            
            # Update progressions
            self.progressions = [ap for ap in self.progressions if worst_element not in ap]
    
    def local_search(self, improve_with_local_search=True): # <- Method required by the pipeline
        """
        Apply local search to fix violations and optionally improve.
        """
        # Step 1: Find all violations
        self._compute_progressions()
        
        # Step 2: Remove elements to fix violations
        self._remove_elements_greedily()
        
        # Step 3: Optionally add more elements
        if improve_with_local_search:
            self._add_elements_greedily()
        
        # Step 4: Recompute progressions (this step is not strictly needed but it's done for consistency)
        self._compute_progressions()
        self.calc_features()
        self.calc_score()
    
    @classmethod
    def _update_class_params(cls, pars): # <- Method required by the pipeline
        pass
    
    @classmethod
    def _save_class_params(cls): # <- Method required by the pipeline
        pass

#### 步骤 2：测试此 DataPoint 结构

如果在启动任何训练之前先测试 DataPoint 结构，您将节省大量时间。

In [4]:
# random data point with N=50
dp1 = ArithmeticProgressionFreeDataPoint(N=50, init=True)

print(f"Universe size: {dp1.N}")
print(f"Number of elements: {dp1.score}")
print(f"Elements: {dp1.get_elements()}")
print(f"Arithmetic progressions: {dp1.progressions}")

elements = dp1.get_elements()
for a, b, c in combinations(elements, 3):
    if 2 * b == a + c:
        print(f"ERROR: Found AP ({a}, {b}, {c})")
        break
else:
    print("Verified: No arithmetic progressions!\n")


# a datapoint with some violations
dp2 = ArithmeticProgressionFreeDataPoint(N=50, init=False)
# in this way, I force the creation of a point with violations
dp2.data[0] = 1
dp2.data[1] = 1
dp2.data[2] = 1

dp2._compute_progressions()
print(f"Arithmetic progressions before running local search: {dp2.progressions}")

dp2.local_search(improve_with_local_search=False)
print(f"Arithmetic progressions after running local search but without local improvement: {dp2.progressions}")
print(f"Score after running local search but without local improvement: {dp2.score}")

dp2.local_search(improve_with_local_search=True)
print(f"Arithmetic progressions after running local search and local improvement: {dp2.progressions}")
print(f"Score after running local search and local improvement: {dp2.score}")

Universe size: 50
Number of elements: 14
Elements: {32, 1, 34, 7, 41, 10, 44, 14, 15, 46, 17, 25, 26, 31}
Arithmetic progressions: []
Verified: No arithmetic progressions!

Arithmetic progressions before running local search: [(0, 1, 2)]
Arithmetic progressions after running local search but without local improvement: []
Score after running local search but without local improvement: 2
Arithmetic progressions after running local search and local improvement: []
Score after running local search and local improvement: 13


#### 步骤 3：创建 Environment

Environment 是 DataPoint 和 tokenizer 之间的桥梁。Tokenizer 是将数学对象转换为 Decoder-only 模型可读的 token 的工具。

In [ ]:
from src.envs.environment import BaseEnvironment
from src.envs.tokenizers import SparseTokenizerSingleInteger


class ArithmeticProgressionFreeEnvironment(BaseEnvironment):
    # this problem lives in N^1, therefore k=1
    # for k=1, are_coordinates_symmetric is not applicable
    # Here we opt for Option 1 and we use an already implemented tokenizer 

    k = 1 # <- has to be specified with Option 1
    are_coordinates_symmetric = False # <- has to be specified with Option 1
    data_class = ArithmeticProgressionFreeDataPoint # <- has to be specified
    
    def __init__(self, params):
        super().__init__(params)
        self.tokenizer = SparseTokenizerSingleInteger(self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS) # <- Our choice of pre-implemented tokenizer 
    
    @staticmethod
    def register_args(parser):
        """
        Register environment parameters.
        """
        parser.add_argument("--N", type=int, default=100, help="Universe size {0, 1, ..., N-1}") #<- Here we only add one parameter specific to the problem, but there could be many others :)


#### 步骤 4：注册 Environment

为了使它能够从主程序运行，您需要在 `src/envs/__init__.py` 中添加一行新代码

```
ENVS = {
    "square": SquareEnvironment,
    "isosceles": IsoscelesEnvironment,
    "sphere": SphereEnvironment,
    "arithmetic_progression": ArithmeticProgressionFreeEnvironment,  # <- 添加此行
}
```

### 示例 2：`N x N` 网格上无 3 点共线

给定一个 `N x N` 网格，找到您可以放置的最大点数，使得没有三个点在同一直线上。

在这个示例中，我将介绍两个额外的想法：
- 该问题在网格的旋转和反射下是不变的，因此我们可以去除正方形 8 种对称性的重复。`make_object_canonical` 控制是否去除仅因这些对称性而不同的重复项。
- 对于给定的网格，有 8 种可能的方式将其转换为其 tokenized 版本。`augment_data_representation` 控制是否使用这 8 种对称性来增强数据。

#### 步骤 1：添加两个特定的函数，使对象规范化并增强数据

In [6]:
import numpy as np
import random

# We pick the lexicographically smallest of the 8 symmetries
def canonical_form_2d(matrix):
    best = None
    best_matrix = None

    # 8 symmetries: 4 rotations × 2 (with/without reflection)
    current = matrix
    for _ in range(4):
        flat = current.flatten().tolist()
        if best is None or flat < best:
            best = flat
            best_matrix = current.copy()

        reflected = np.flip(current, axis=1)
        flat = reflected.flatten().tolist()
        if flat < best:
            best = flat
            best_matrix = reflected.copy()

        current = np.rot90(current)

    return best_matrix


# We pick one of the 8 symmetries at random
def random_symmetry_2d(matrix):
    k = random.randint(0, 3)
    result = np.rot90(matrix, k)

    if random.randint(0, 1):
        result = np.flip(result, axis=1)

    return result.copy()


#### 步骤 2：构建 DataPoint 结构

In [ ]:
import numpy as np
from itertools import combinations
from src.envs.environment import DataPoint


class CollinearDataPoint(DataPoint):
    """
    Represents a set of points on an N x N grid with no three collinear.
    
    Attributes:
        N: Grid size
        data: N x N binary matrix, data[i,j] = 1 means point (i,j) is selected
        collinear_triplets: List of collinear triplets (violations, needed for local search)
        score: Number of points if valid, -1 otherwise
        features: String representation
    """
    
    MAKE_OBJECT_CANONICAL = False # <- Additional argument specific to the problem
    
    def __init__(self, N, init=False):
        self.N = N
        self.data = np.zeros((N, N), dtype=np.uint8)
        self.collinear_triplets = []
        
        if init: # <- Required by the pipeline; If true, we want to create an element at random. This will be used during the process to differenciate between generation and decoding of the output of the neural network.
            self._add_points_greedily() # <- Method to create a valid construction at random.
            if self.MAKE_OBJECT_CANONICAL:
                self.data = canonical_form_2d(self.data)
            self.calc_features() # <- Required by the pipeline;
            self.calc_score() # <- Required by the pipeline;
    
    def get_points(self): # <- Method specific to the problem
        """Return list of selected points as (x, y) tuples."""
        return [(i, j) for i in range(self.N) for j in range(self.N) if self.data[i, j] == 1]
    
    def calc_score(self): # <- Method required by the pipeline
        if len(self.collinear_triplets) > 0:
            self.score = -1
        else:
            self.score = int(self.data.sum())
    
    def calc_features(self): # <- Optional method used by the pipeline
        """
        Create a string representation of the configuration for deduplication.
        """
        w = []
        for i in range(self.N):
            for j in range(self.N):
                w.append(self.data[i, j])
        self.features = ",".join(map(str, w))
    
    def _are_collinear(self, p1, p2, p3): # <- Method specific to the problem
        """Check if three points are collinear using cross product."""
        x1, y1 = p1
        x2, y2 = p2
        x3, y3 = p3
        cross = (x2 - x1) * (y3 - y1) - (x3 - x1) * (y2 - y1)
        return cross == 0
    
    def _would_create_collinear(self, new_point, existing_points): # <- Method specific to the problem
        """
        Check if adding new_point would create a collinear triplet with any pair of existing points.
        """
        for p1, p2 in combinations(existing_points, 2):
            if self._are_collinear(p1, p2, new_point):
                return True
        return False
    
    def _add_points_greedily(self): # <- Method specific to the problem
        """
        Greedily add points to the grid while avoiding collinear triplets.
        We iterate through all grid positions in random order and add a point if it doesn't create a collinear triplet.
        """
        all_positions = [(i, j) for i in range(self.N) for j in range(self.N)]
        np.random.shuffle(all_positions)
        
        current_points = self.get_points()
        
        for pos in all_positions:
            if self.data[pos] == 0:
                if not self._would_create_collinear(pos, current_points):
                    self.data[pos] = 1
                    current_points.append(pos)
    
    def _compute_collinear_triplets(self): # <- Method specific to the problem
        """Find all collinear triplets in the current configuration."""
        current_points = self.get_points()
        self.collinear_triplets = []
        
        for p1, p2, p3 in combinations(current_points, 3):
            if self._are_collinear(p1, p2, p3):
                triplet = tuple(sorted([p1, p2, p3]))
                self.collinear_triplets.append(triplet)
    
    def _remove_points_greedily(self): # <- Method specific to the problem
        """
        Greedily remove points to eliminate all collinear triplets.
        At each step, remove the point that appears in the most triplets.
        """
        while self.collinear_triplets:
            # Count how many triplets each point appears in
            point_count = {}
            for triplet in self.collinear_triplets:
                for point in triplet:
                    point_count[point] = point_count.get(point, 0) + 1
            
            # Find the point in the most triplets
            worst_point = max(point_count, key=point_count.get)
            
            # Remove it
            self.data[worst_point] = 0
            
            # Update triplets (remove all triplets containing this point)
            self.collinear_triplets = [t for t in self.collinear_triplets if worst_point not in t]
    
    def local_search(self, improve_with_local_search=True): # <- Method requiremd by the pipeline
        """
        Apply local search to fix violations and optionally improve the solution.
        """
        # Step 1: Find all violations
        self._compute_collinear_triplets()
        
        # Step 2: Remove points to fix violations
        self._remove_points_greedily()
        
        # Step 3: Optionally try to add more points
        if improve_with_local_search:
            self._add_points_greedily()
        
        # Step 4: Recompute progressions (this step is not strictly needed but it's done for consistency)
        self._compute_collinear_triplets()
        self.calc_features()
        self.calc_score()
    
    @classmethod
    def _update_class_params(cls, pars): # <- Method requiremd by the pipeline
        """Update class-level parameters (used for multiprocessing)."""
        cls.MAKE_OBJECT_CANONICAL = pars
    
    @classmethod
    def _save_class_params(cls): # <- Method requiremd by the pipeline
        """Save class-level parameters (used for multiprocessing)."""
        return cls.MAKE_OBJECT_CANONICAL


#### 步骤 3：测试新的数据点类

In [8]:
dp1 = CollinearDataPoint(N=10, init=True)

print(f"Grid size: {dp1.N} x {dp1.N}")
print(f"Number of points: {dp1.score}")
print(f"Points: {dp1.get_points()}")
print(f"Collinear triplets: {dp1.collinear_triplets}\n")


# a datapoint with some violations
dp2 = CollinearDataPoint(N=10, init=False)
# in this way, I force the creation of a point with violations
dp2.data[0, 0] = 1
dp2.data[1, 1] = 1
dp2.data[2, 2] = 1

dp2._compute_collinear_triplets()
print(f"Collinear triplets before running local search: {dp2.collinear_triplets}")

dp2.local_search(improve_with_local_search=False)
print(f"Collinear triplets after running local search but without local improvement: {dp2.collinear_triplets}")
print(f"Score after running local search but without local improvement: {dp2.score}")

dp2.local_search(improve_with_local_search=True)
print(f"Collinear triplets after running local search and local improvement: {dp2.collinear_triplets}")
print(f"Score after running local search and local improvement: {dp2.score}")

Grid size: 10 x 10
Number of points: 15
Points: [(0, 7), (0, 9), (1, 1), (1, 2), (3, 8), (4, 4), (4, 9), (5, 2), (6, 3), (6, 8), (7, 3), (7, 5), (8, 0), (8, 4), (9, 7)]
Collinear triplets: []

Collinear triplets before running local search: [((0, 0), (1, 1), (2, 2))]
Collinear triplets after running local search but without local improvement: []
Score after running local search but without local improvement: 2
Collinear triplets after running local search and local improvement: []
Score after running local search and local improvement: 16


#### 步骤 4：创建 Environment 类

In [ ]:
from src.envs.environment import BaseEnvironment
from src.envs.tokenizers import SparseTokenizerSingleInteger, SparseTokenizerSequenceKTokens, DenseTokenizer
from src.utils import bool_flag


class CollinearEnvironment(BaseEnvironment):
    # this problem lives in N^2, therefore k=2
    # (i, j) or (j, i) represents two distinct points in the grid, therefore are_coordinates_symmetric=False    
    # Here we opt for Option 1 and we use *several* already implemented tokenizers 

    k = 2 # <- has to be specified with Option 1
    are_coordinates_symmetric = False # <- has to be specified with Option 1
    data_class = CollinearDataPoint # <- has to be specified
    
    def __init__(self, params):
        super().__init__(params)
        self.data_class.MAKE_OBJECT_CANONICAL = params.make_object_canonical
        encoding_augmentation = random_symmetry_2d if params.augment_data_representation else None
        if params.encoding_tokens == "single_integer":
            self.tokenizer = SparseTokenizerSingleInteger(
                self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS, encoding_augmentation=encoding_augmentation
            ) # <- Possible choice of pre-implemented tokenizer #1
        elif params.encoding_tokens == "sequence_k_tokens":
            self.tokenizer = SparseTokenizerSequenceKTokens(
                self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS, encoding_augmentation=encoding_augmentation
            ) # <- Possible choice of pre-implemented tokenizer #2
        elif params.encoding_tokens == "adjacency":
            self.tokenizer = DenseTokenizer(
                self.data_class, params.N, self.k, self.are_coordinates_symmetric, self.SPECIAL_SYMBOLS, pow2base=params.pow2base, encoding_augmentation=encoding_augmentation
            ) # <- Possible choice of pre-implemented tokenizer #3
        else:
            raise ValueError(f"Invalid encoding: {params.encoding_tokens}")
    
    @staticmethod
    def register_args(parser):
        parser.add_argument("--N", type=int, default=10, help="Grid size N x N")
        parser.add_argument("--encoding_tokens", type=str, default="single_integer", help="single_integer/sequence_k_tokens/adjacency")
        parser.add_argument("--make_object_canonical", type=bool_flag, default="false", help="sort the grid by symmetry")
        parser.add_argument("--augment_data_representation", type=bool_flag, default="false", help="augment the data representation with predefined function")
        parser.add_argument("--pow2base", type=int, default=1, help="Bits per token for adjacency encoding")

#### 步骤 5：注册 Environment

与前面的环境类似，您需要在 `src/envs/__init__.py` 中添加一行新代码

```
ENVS = {
    "square": SquareEnvironment,
    "isosceles": IsoscelesEnvironment,
    "sphere": SphereEnvironment,
    "collinear": CollinearEnvironment,  # 添加此行
}
```